In [1]:
import requests
import asyncio
import pandas as pd
import time
import urllib3
from pathlib import Path
urllib3.disable_warnings(urllib3.exceptions.InsecureRequestWarning)
import discord

import credentials as cr
from gapi.riot_api import RiotAPI
from gapi.live_client_api import LiveClientAPI, event_monitoring, event_parsing


# Use Riot game API for after game data

In [ ]:
game_name = cr.game_name # your pseudo
tag_line = cr.tag_line # your tag

bot = RiotAPI(region='europe', api_key=cr.API_KEY) # class needs region and api key

puuid = bot.get_puuid(game_name, tag_line)['puuid'] # get player primary key from pseudo and tag

history = bot.get_match_history(puuid) # get match history from player primary key (puuid)

bot.get_match_data(history[0])# get match data from match_id

# Use Live Client API for real time data

In [ ]:
import requests
import pandas as pd
import time
import urllib3
urllib3.disable_warnings(urllib3.exceptions.InsecureRequestWarning)
import credentials as cr
import discord
from gapi.live_client_api import LiveClientAPI
import asyncio
import nacl


############################################################################ event   -> gapi/live_events.py

def event_parsing(event, data):
    event_name = event['EventName']
    assisters = event.get('Assisters', [])
    killer = event.get('KillerName', None)
    victime = event.get('VictimName', None)

    data.append({"eventid": event['EventID'],
            "event_name": event_name,
            "assisters": assisters,
            "killer": killer,
            "victime": victime})
    
    return data


async def event_monitoring(crew:list, response, last_event_id, data, bot:discord.Client):
    player_died = False
    
    
    events = response['events']['Events']

    for event in events:
        if event['EventID'] <= last_event_id:
            continue

        print("New event:", event["EventID"], event["EventName"])

        event_parsing(event, data)

        if event["EventName"] == "ChampionKill":
            victim = event.get("VictimName")
            killer = event.get("KillerName")
            assisters = event.get("Assisters", [])

            if victim in crew:
                print('player dead')
                player_died = True

            if killer in crew:
                print("A crew member has made a kill")

            if any(assister in crew for assister in assisters):
                print("A crew member has made an assist")

        last_event_id = event["EventID"]

    return last_event_id, data, player_died


############################################################################ audio   -> bot/audio.py


async def play_death_sound(bot:discord.Client ,guild_id:int, voice_channel_id:int):
    
    guild = bot.get_guild(guild_id) # server
    if guild is None :
        print('Cant find discord server')
        return

    channel = guild.get_channel(voice_channel_id) # channel
    if channel is None:
        print('Cant find discord channel')
        return

    voice_client = guild.voice_client

    if voice_client is None: # bot is not connected to a voice chat
        voice_client = await channel.connect()

    elif voice_client.channel.id != channel.id: # bot is connected to another voicechat
        await voice_client.move_to(channel)

    if voice_client.is_playing(): # if already playing a song, stops  (keeping it ??)
        voice_client.stop()

    source = discord.FFmpegPCMAudio(str(r'bot\sounds\death.mp3'),executable=r'tools\ffmpeg.exe')

    voice_client.play(source)




############################################################################ loop  -> robot.py

intents = discord.Intents.all()
bot = discord.Client(intents=intents)

@bot.event
async def on_ready():
    serv = cr.discord_channel # guild or server id
    chan = cr.discord_salon_vocal # channel id 

    print(f"Bot connecté : {bot.user}")

    guild = bot.get_guild(serv)
    channel = bot.get_channel(chan)

    print("Serveur :", guild)
    print("Salon :", channel)



crew = [cr.game_name]
data = []
last_event_id = -1

async def monitoring_loop():
    
    await bot.wait_until_ready()
    global last_event_id, data

    while True:
        try:
            response = LiveClientAPI().get_live_data()
            if "ok" in response :
                print(response)
                break
            

            last_event_id, data, p = await event_monitoring(bot=bot, crew=crew,response=response ,last_event_id=last_event_id ,data=data,)   

            if p is True:
                await play_death_sound(bot=bot, guild_id=cr.guild_id, voice_channel_id=cr.voc_channel_id)


        except Exception as error:
            print("Erreur pendant le monitoring :", error)

        await asyncio.sleep(1)


async def main():
    async with bot:
        # Starting monitoring as a parallel task
        monitoring_task = asyncio.create_task(monitoring_loop())

        try:
            # starting the bot and staying active while the bot is up
            await bot.start(cr.discord_bot_token)

        finally:
            # if bot stops, monitoring stops
            monitoring_task.cancel()


await main() 


New event: 0 GameStart
New event: 1 ChampionKill
Cant find discord server
New event: 2 MinionsSpawning
New event: 3 TurretKilled
New event: 4 FirstBrick


# Testing discord.py

In [ ]:
serv = cr.discord_channel
chan = cr.discord_salon_vocal

intents = discord.Intents.all() # all kind of permissions
bot = discord.Client(intents=intents) # gived to the bot


""" more scalable
BASE_DIR = Path(__file__).resolve().parent

# ffmpeg source
FFMPEG_SOURCE = BASE_DIR / "tools" / "ffmpeg.exe"

# mp3 source
DEATH_SOUND = BASE_DIR / "bot" / "sounds" / "death.mp3"
"""

FFMPEG_SOURCE = r"tools\ffmpeg.exe"
DEATH_SOUND = r"bot\sounds\death.mp3"

source = discord.FFmpegPCMAudio(str(DEATH_SOUND),executable=FFMPEG_SOURCE)



# bot goes to the right place
guild = bot.get_guild(serv) 
channel = bot.get_channel(chan)

# bot is on
@bot.event
async def on_ready():
    print(f"Bot connecté : {bot.user}")

    guild = bot.get_guild(serv)
    channel = bot.get_channel(chan)

    print("Serveur :", guild)
    print("Salon :", channel)

await bot.start(cr.discord_bot_token)





[2026-07-12 11:05:33] [INFO    ] discord.player: ffmpeg process 34976 has not terminated. Waiting to terminate...
[2026-07-12 11:05:33] [INFO    ] discord.player: ffmpeg process 34976 should have terminated with a return code of 1.
[2026-07-12 11:05:33] [INFO    ] discord.client: logging in using static token
[2026-07-12 11:05:34] [INFO    ] discord.gateway: Shard ID None has connected to Gateway (Session ID: 432092ae87fa21a23bab964c0fb2bac7).


Bot connecté : robot.py#1099
Serveur : Serveur de test
Salon : Général


# Notes

In [ ]:
""" 
activePlayer = player heberging the bot:

data["activePlayer"]["championStats"]["currentHealth"]
data["activePlayer"]["championStats"]["maxHealth"]
data["activePlayer"]["championStats"]["attackDamage"]
data["activePlayer"]["championStats"]["armor"]
data["activePlayer"]["abilities"]
data["activePlayer"]["currentGold"]
data["activePlayer"]["level"]
data["activePlayer"]["riotId"]


allPlayers = all players from the game:

data["allPlayers"]player["riotId"]
data["allPlayers"]player["championName"]
data["allPlayers"]player["isDead"]
data["allPlayers"]player["respawnTimer"]
data["allPlayers"]player["scores"]["kills"]
data["allPlayers"]player["scores"]["deaths"]
data["allPlayers"]player["scores"]["assists"]
data["allPlayers"]player["scores"]["creepScore"]
data["allPlayers"]player["items"]
data["allPlayers"]player["team"]
"""

définir une class genre crew
le but c'est de check les données en temps réel dans allPlayers pour tous les riotId in crew


idée event:
- si tu es très lowlife et que tu ne meurs pas (exemple hp<10 pendant 10 sec) => survivant
- lancer audio à chaque kill, death assist, win ou loose


In [ ]:
# backup
async def play_death_sound(guild_id:int, voice_channel_id:int):
    
    guild = bot.get_guild(guild_id) # server
    if guild is None :
        print('Cant find discord server')

    channel = guild.get_channel(voice_channel_id) # channel
    if channel is None:
        print('Cant find discord channel')


    voice_client = guild.voice_client

    if voice_client is None: # bot is not connected to a voice chat
        voice_client = await channel.connect()

    elif voice_client.channel.id != channel.id: # bot is connected to another voicechat
        await voice_client.move_to(channel)

    if voice_client.is_playing(): # if already playing a song, stops  (keeping it ??)
        voice_client.stop()

    source = discord.FFmpegPCMAudio(str(DEATH_SOUND))

    voice_client.play(
        source,
        after=lambda error: print(f"Erreur audio : {error}") if error else None,
    )